# Course 5 - Applied Social Network Analysis in Python
## Assignment 3: Centrality Measures and Link Analysis

### Overview
This assignment explores centrality measures on two real networks. Part 1 uses a university friendship network to find which node best spreads information under different constraints. Part 2 uses a political blog network to apply PageRank and HITS (Hyperlink-Induced Topic Search) algorithms.

### Learning Objectives
- Compute degree, closeness, and betweenness centrality for nodes
- Select nodes based on centrality criteria for different information-spreading strategies
- Apply the PageRank algorithm to rank nodes in a directed graph
- Apply the HITS algorithm to find hub and authority scores
- Interpret the difference between hubs (good linkers) and authorities (well-cited)

### Datasets
- **`assets/friendships.gml`** — Undirected friendship network (G1), university department
- **`assets/blogs.gml`** — Directed political blog network (G2), nodes=blogs, edges=hyperlinks

### Assignment Structure

**Part 1 — Friendship Network Centrality (G1)**
| Question | Centrality Used | Reasoning |
|----------|----------------|-----------|
| Q1 | Degree, Closeness, Betweenness of node 100 | Baseline centrality measures |
| Q2 | Degree centrality | Voucher travels 1 hop — maximize direct neighbors |
| Q3 | Closeness centrality | Voucher travels unlimited — minimize average distance to all |
| Q4 | Betweenness centrality | Competitor removes bridges — node on most shortest paths |

**Part 2 — Blog Network Link Analysis (G2)**
| Question | Algorithm | Task |
|----------|-----------|------|
| Q5 | PageRank (alpha=0.85) | PageRank of 'realclearpolitics.com' |
| Q6 | PageRank (alpha=0.85) | Top 5 nodes by PageRank |
| Q7 | HITS | Hub and authority scores of 'realclearpolitics.com' |
| Q8 | HITS | Top 5 nodes by hub score |
| Q9 | HITS | Top 5 nodes by authority score |

### Key Concepts
- **Degree centrality:** fraction of nodes a node is connected to (local influence)
- **Closeness centrality:** inverse of average distance to all other nodes (efficiency of information spread)
- **Betweenness centrality:** fraction of shortest paths passing through the node (brokerage/bridge role)
- **PageRank:** importance based on the importance of incoming links (recursive)
- **HITS Hubs:** nodes that link to many high-authority pages; **HITS Authorities:** nodes linked from many high-hub pages

## Part 1

Answer questions 1-4 using the network `G1`, a network of friendships at a university department. Each node corresponds to a person, and an edge indicates friendship. 

*The network has been loaded as networkx graph object `G1`.*

In [1]:
import networkx as nx

G1 = nx.read_gml('assets/friendships.gml')

### What This Code Does
Loads the friendship network from a GML (Graph Markup Language) file. GML is a standard text format for describing graphs with node and edge attributes. `G1` is an undirected graph where each node has an integer ID and edges represent declared friendships. The network will be used to demonstrate different centrality-based strategies for information dissemination.

### Question 1

Find the degree centrality, closeness centrality, and betweeness centrality of node 100.

*This function should return a tuple of floats `(degree_centrality, closeness_centrality, betweenness_centrality)`.*

In [2]:
def answer_one():
    # Compute three centrality measures for node 100 in the friendship network
    degree_centrality = nx.degree_centrality(G1)[100]
    closeness_centrality = nx.closeness_centrality(G1)[100]
    betweenness_centrality = nx.betweenness_centrality(G1)[100]
    return (degree_centrality, closeness_centrality, betweenness_centrality)

ans_one = answer_one()
assert type(ans_one) == tuple, "You must return a tuple"
ans_one

(0.0026501766784452294, 0.2654784240150094, 7.142902633244772e-05)

In [3]:
ans_one = answer_one()
assert type(ans_one) == tuple, "You must return a tuple"


### Use centrality measures to answer questions 2-4

### Question 2

Suppose you are employed by an online shopping website and are tasked with selecting one user in network G1 to send an online shopping voucher to. We expect that the user who receives the voucher will send it to their friends in the network.  You want the voucher to reach as many nodes as possible. The voucher can be forwarded to multiple users at the same time, but the travel distance of the voucher is limited to one step, which means if the voucher travels more than one step in this network, it is no longer valid. Apply your knowledge in network centrality to select the best candidate for the voucher. 

*This function should return an integer, the chosen node.*

In [4]:
def answer_two():
    # Voucher travels at most 1 hop: maximize direct neighbors -> use degree centrality
    # The node with highest degree centrality reaches the most people in one step
    degree_centrality = nx.degree_centrality(G1)
    return max(degree_centrality, key=degree_centrality.get)

ans_two = answer_two()
ans_two

105

In [5]:
ans_two = answer_two()


### Question 3

Now the limit of the voucher’s travel distance has been removed. Because the network is connected, regardless of who you pick, every node in the network will eventually receive the voucher. However, we now want to ensure that the voucher reaches nodes as quickly as possible (i.e. in the fewest number of hops). How will you change your selection strategy? Write a function to tell us who is the best candidate in the network under this condition.

*This function should return an integer, the chosen node.*

In [6]:
def answer_three():
    # Voucher travels unlimited hops: minimize average path length -> use closeness centrality
    # The node with highest closeness centrality is on average closest to all other nodes
    closeness_centrality = nx.closeness_centrality(G1)
    return max(closeness_centrality, key=closeness_centrality.get)

ans_three = answer_three()
ans_three

23

In [7]:
ans_three = answer_three()


### Question 4

Assume the restriction on the voucher’s travel distance is still removed, but now a competitor has developed a strategy to remove a person from the network in order to disrupt the distribution of your company’s voucher. You competitor plans to remove people who act as bridges in the network. Identify the best possible person to be removed by your competitor?

*This function should return an integer, the chosen node.*

In [8]:
def answer_four():
    # Competitor targets bridges/bottlenecks -> use betweenness centrality
    # The node with highest betweenness is on the most shortest paths (removing it disrupts most routes)
    betweenness_centrality = nx.betweenness_centrality(G1)
    return max(betweenness_centrality, key=betweenness_centrality.get)

ans_four = answer_four()
ans_four

333

In [9]:
ans_four = answer_four()


## Part 2

`G2` is a directed network of political blogs, where nodes correspond to a blog and edges correspond to links between blogs. Use your knowledge of PageRank and HITS to answer Questions 5-9.

### What This Code Does
Loads the political blog network `G2` in GML format. This is a directed graph where:
- **Nodes** are political blogs (identified by blog URL string)
- **Directed edges** represent hyperlinks from one blog to another
- The direction of links is significant: PageRank and HITS treat in-links differently from out-links

This network is the basis for the PageRank and HITS analyses in Questions 5-9.

In [10]:
G2 = nx.read_gml('assets/blogs.gml')

### Question 5

Apply the Scaled Page Rank Algorithm to this network. Find the Page Rank of node 'realclearpolitics.com' with damping value 0.85.

*This function should return a float.*

In [11]:
def answer_five():
    # PageRank with damping factor alpha=0.85 (probability of following a link vs random jump)
    pr = nx.pagerank(G2, alpha=0.85)
    return pr['realclearpolitics.com']

ans_five = answer_five()
ans_five

0.004636694781649098

In [12]:
ans_five = answer_five()


### Question 6

Apply the Scaled Page Rank Algorithm to this network with damping value 0.85. Find the 5 nodes with highest Page Rank. 

*This function should return a list of the top 5 blogs in desending order of Page Rank.*

In [13]:
def answer_six():
    # Return the 5 nodes with highest PageRank scores (descending order)
    pr = nx.pagerank(G2, alpha=0.85)
    return sorted(pr, key=pr.get, reverse=True)[:5]

ans_six = answer_six()
assert type(ans_six) == list, "You must return a list"
ans_six

['dailykos.com',
 'atrios.blogspot.com',
 'instapundit.com',
 'blogsforbush.com',
 'talkingpointsmemo.com']

In [14]:
ans_six = answer_six()
assert type(ans_six) == list, "You must return a list"


### Question 7

Apply the HITS Algorithm to the network to find the hub and authority scores of node 'realclearpolitics.com'. 

*Your result should return a tuple of floats `(hub_score, authority_score)`.*

In [15]:
def answer_seven():
    # HITS: hub scores reflect how many good authorities a page links to
    #       authority scores reflect how many good hubs link to the page
    hubs, authorities = nx.hits(G2)
    return (hubs['realclearpolitics.com'], authorities['realclearpolitics.com'])

ans_seven = answer_seven()
assert type(ans_seven) == tuple, "You must return a tuple"
ans_seven

(0.0003243556140278726, 0.003918957644934246)

In [16]:
ans_seven = answer_seven()
assert type(ans_seven) == tuple, "You must return a tuple"


### Question 8 

Apply the HITS Algorithm to this network to find the 5 nodes with highest hub scores.

*This function should return a list of the top 5 blogs in desending order of hub scores.*

In [17]:
def answer_eight():
    # Top 5 hub nodes — pages that are good aggregators/linkers to authority pages
    hubs, authorities = nx.hits(G2)
    return sorted(hubs, key=hubs.get, reverse=True)[:5]

ans_eight = answer_eight()
assert type(ans_eight) == list, "You must return a list"
ans_eight

['politicalstrategy.org',
 'madkane.com/notable.html',
 'liberaloasis.com',
 'stagefour.typepad.com/commonprejudice',
 'bodyandsoul.typepad.com']

In [18]:
ans_eight = answer_eight()
assert type(ans_eight) == list, "You must return a list"


### Question 9 

Apply the HITS Algorithm to this network to find the 5 nodes with highest authority scores.

*This function should return a list of the top 5 blogs in desending order of authority scores.*

In [19]:
def answer_nine():
    # Top 5 authority nodes — pages that are well-cited by hub pages
    hubs, authorities = nx.hits(G2)
    return sorted(authorities, key=authorities.get, reverse=True)[:5]

ans_nine = answer_nine()
assert type(ans_nine) == list, "You must return a list"
ans_nine

['dailykos.com',
 'talkingpointsmemo.com',
 'atrios.blogspot.com',
 'washingtonmonthly.com',
 'talkleft.com']

In [20]:
ans_nine = answer_nine()
assert type(ans_nine) == list, "You must return a list"
